In [1]:
"""
Interfaz interactiva.
"""

import time
import csv
import os
import sys
from inspire_rh56dftp import InspireHand

try:
    from IPython.display import clear_output
    ENTORNO_JUPYTER = True
except ImportError:
    ENTORNO_JUPYTER = False

IP_MANO = "192.168.123.211"
NOMBRES = ["Meñique", "Anular", "Corazón", "Índice", "Flexión pulgar", "Rotación pulgar"]

# --- CONFIGURACIÓN DE DIRECTORIOS Y ARCHIVOS POR MODO ---
DIRECTORIO_GUARDADO = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Gestos")
DIRECTORIO_RUTINAS = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Rutinas")

ARCHIVOS_CSV = [
    "gestos_0_solo_angulos.csv",
    "gestos_1_angulos_fuerza.csv",
    "gestos_2_angulos_velocidad.csv",
    "gestos_3_completos.csv"
]

ARCHIVOS_RUTINAS_CSV = [
    "rutinas_0_solo_angulos.csv",
    "rutinas_1_angulos_fuerza.csv",
    "rutinas_2_angulos_velocidad.csv",
    "rutinas_3_completos.csv"
]

CABECERAS_RUTINAS = ["Nombre_Rutina", "Secuencia_Indices", "Secuencia_Nombres"]

def obtener_cabeceras(modo_idx):
    """Genera las cabeceras del CSV de gestos dependiendo del modo actual"""
    base = ["Nombre_Gesto"] + [f"Ang_{n}" for n in NOMBRES]
    if modo_idx == 1:
        return base + [f"Fue_{n}" for n in NOMBRES]
    elif modo_idx == 2:
        return base + [f"Vel_{n}" for n in NOMBRES]
    elif modo_idx == 3:
        return base + [f"Fue_{n}" for n in NOMBRES] + [f"Vel_{n}" for n in NOMBRES]
    return base

# --- POSTURAS Y PERFILES BASE ---
ABIERTO = [1000, 1000, 1000, 1000, 1000, 1000]
FUERZA_SEGURA = [200, 200, 200, 200, 200, 200]
MAX_FUERZA = [1000, 1000, 1000, 1000, 1500, 1000]
MAX_VELOCIDAD = [1000, 1000, 1000, 1000, 1000, 1000]

MODOS_NOMBRES = [
    "Solo Ángulos (Fuerza Segura)",
    "Ángulos + Fuerza configurable",
    "Ángulos + Velocidad configurable",
    "Ambas (Ángulo + Fuerza + Velocidad)"
]

# Direcciones Decimales de los Registros Modbus Corregidas
REG_ANGLE_SET = 1486
REG_DEFAULT_FORCE_SET = 1044
REG_FORCE_SET = 1498
REG_SPEED_SET = 1522
REG_ANGLE_ACT = 1546
REG_FORCE_ACT = 1582

def limpiar_pantalla():
    """Limpia la pantalla evitando bloqueos en Jupyter"""
    if ENTORNO_JUPYTER:
        clear_output(wait=False)
    else:
        os.system('cls' if os.name == 'nt' else 'clear')

def leer_estado_completo(mano):
    """Lee los sensores físicos existentes: Ángulos (1546) y Fuerzas (1582), 
    convirtiéndolos a enteros con signo para evitar el desajuste a 65535."""
    try:
        angulos_raw = mano.read_holding(REG_ANGLE_ACT, 6)
        angulos = [v if v < 32768 else v - 65536 for v in angulos_raw]
    except Exception:
        angulos = ABIERTO.copy()
        
    try:
        fuerzas_raw = mano.read_holding(REG_FORCE_ACT, 6)
        # Convierte uint16 a int16 y normaliza offsets negativos a 0 (rango real 0-1000 / 1500)
        fuerzas = [max(0, v if v < 32768 else v - 65536) for v in fuerzas_raw]
    except Exception:
        fuerzas = MAX_FUERZA.copy()
        
    return list(angulos), list(fuerzas)

def enviar_parametros(mano, angulos, fuerzas=None, velocidades=None, default_fuerzas=None, timeout=5.0, tolerancia=40):
    """Inyecta datos utilizando los registros Modbus correspondientes."""
    if default_fuerzas is not None:
        clamped_def_fuerzas = [max(0, min(1500, int(v))) for v in default_fuerzas]
        try:
            mano._write_and_ack(REG_DEFAULT_FORCE_SET, clamped_def_fuerzas)
            time.sleep(0.05)
        except Exception as e:
            print(f" ⚠ Error al enviar DEFAULT_FORCE_SET (1044): {e}")

    if fuerzas is not None:
        clamped_fuerzas = [max(0, min(1500, int(v))) for v in fuerzas]
        try:
            mano._write_and_ack(REG_FORCE_SET, clamped_fuerzas)
            time.sleep(0.05)
        except Exception as e:
            print(f" ⚠ Error al enviar fuerzas (1498): {e}")

    if velocidades is not None:
        clamped_velocidades = [max(0, min(1000, int(v))) for v in velocidades]
        try:
            mano._write_and_ack(REG_SPEED_SET, clamped_velocidades)
            time.sleep(0.05)
        except Exception as e:
            print(f" ⚠ Error al enviar velocidades (1522): {e}")

    mano.send_raw(angulos)
    
    inicio = time.time()
    
    while time.time() - inicio < timeout:
        try:
            angulos_act = list(mano.read_holding(REG_ANGLE_ACT, 6))
            if all(abs(act - obj) <= tolerancia for act, obj in zip(angulos_act, angulos)):
                tiempo_real = time.time() - inicio
                print(f" ✓ Gesto completado en {tiempo_real:.2f}s.")
                return True
        except Exception:
            pass 
        time.sleep(0.1)
        
    print(f" ⚠ Tiempo agotado ({timeout}s) para alcanzar la pose exacta.")
    return False

def recoger_matriz(nombre_medida, valor_mantener=-1, vector_actual=None):
    print(f"\nIngrese valores para {nombre_medida} (Rango 0-1000).")
    if vector_actual is not None:
        print(f"Ingrese {valor_mantener} para mantener el valor actual de cada dedo.")
    
    matriz = []
    for idx, nombre in enumerate(NOMBRES):
        lectura = f" [Actual: {vector_actual[idx]}]" if vector_actual is not None else ""
        sys.stdout.flush()
        time.sleep(0.1)
        try:
            val = int(input(f"{nombre_medida} para {nombre}{lectura}: "))
            if val == valor_mantener and vector_actual is not None:
                matriz.append(vector_actual[idx])
            else:
                if nombre_medida == "Fuerza límite" and idx == 4 and val > 1000:
                    val = min(val, 1500)
                else:
                    val = max(0, min(1000, val))
                matriz.append(val)
        except ValueError:
            matriz.append(vector_actual[idx] if vector_actual is not None else 1000)
    return matriz

# --- FUNCIONES AUXILIARES PARA GESTOS Y RUTINAS ---
def cargar_gestos_csv(ruta_csv):
    """Carga y devuelve la lista de gestos guardados en el CSV del modo actual."""
    if not os.path.exists(ruta_csv):
        return None
    gestos = []
    with open(ruta_csv, mode='r', newline='') as file:
        reader = csv.reader(file)
        try:
            next(reader)
        except StopIteration:
            pass
        for row in reader:
            if row:
                gestos.append(row)
    return gestos

def ejecutar_gesto_desde_fila(mano, gesto_row, modo_actual_idx):
    """Desempaqueta una fila del CSV según el modo activo y envía los parámetros a la mano."""
    postura_ang = [int(val) for val in gesto_row[1:7]]
    def_f_imp = FUERZA_SEGURA.copy() if modo_actual_idx == 0 else None
    f_imp = None if modo_actual_idx == 0 else (
        [int(val) for val in gesto_row[7:13]] if modo_actual_idx in [1, 3] else MAX_FUERZA.copy()
    )
    v_idx_start = 13 if modo_actual_idx == 3 else 7
    v_imp = (
        [int(val) for val in gesto_row[v_idx_start:v_idx_start+6]]
        if modo_actual_idx in [2, 3] else MAX_VELOCIDAD.copy()
    )
    enviar_parametros(mano, postura_ang, f_imp, v_imp, default_fuerzas=def_f_imp, timeout=5.0)
    return v_imp

def ejecutar_secuencia_gestos(mano, indices, gestos_guardados, modo_actual_idx):
    """Ejecuta una secuencia de gestos optimizando los tiempos de espera entre pasos."""
    ultima_vel = MAX_VELOCIDAD.copy()
    print("\n=== INICIANDO RUTINA SECUENCIAL ===")
    
    for idx in indices:
        if 0 <= idx < len(gestos_guardados):
            gesto_row = gestos_guardados[idx]
            print(f"\n>> Paso de rutina: Ejecutando '{gesto_row[0]}' (Índice {idx})...")
            postura_ang = [int(val) for val in gesto_row[1:7]]
            def_f_imp = FUERZA_SEGURA.copy() if modo_actual_idx == 0 else None
            f_imp = None if modo_actual_idx == 0 else (
                [int(val) for val in gesto_row[7:13]] if modo_actual_idx in [1, 3] else MAX_FUERZA.copy()
            )
            v_idx_start = 13 if modo_actual_idx == 3 else 7
            v_imp = (
                [int(val) for val in gesto_row[v_idx_start:v_idx_start+6]]
                if modo_actual_idx in [2, 3] else MAX_VELOCIDAD.copy()
            )
            enviar_parametros(mano, postura_ang, f_imp, v_imp, default_fuerzas=def_f_imp, timeout=5.0)
            time.sleep(0.3)
        else:
            print(f"⚠ Índice {idx} ignorado (fuera de rango).")
            
    print("\n✓ Ejecución de rutina finalizada exitosamente.")
    return ultima_vel

def exportar_rutina_csv(ruta_rutinas_csv, nombre_rutina, indices, gestos_guardados):
    """Exporta una secuencia de gestos al archivo CSV de rutinas correspondiente al modo actual."""
    archivo_existe = os.path.exists(ruta_rutinas_csv)
    secuencia_indices_str = ",".join(str(i) for i in indices)
    nombres_pasos = [
        gestos_guardados[i][0] if 0 <= i < len(gestos_guardados) else f"Idx_{i}"
        for i in indices
    ]
    secuencia_nombres_str = " -> ".join(nombres_pasos)

    with open(ruta_rutinas_csv, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not archivo_existe:
            writer.writerow(CABECERAS_RUTINAS)
        writer.writerow([nombre_rutina, secuencia_indices_str, secuencia_nombres_str])

    print(f" ✓ Rutina '{nombre_rutina}' exportada exitosamente en '{os.path.basename(ruta_rutinas_csv)}'.")

def cargar_rutinas_csv(ruta_rutinas_csv):
    """Lee y devuelve las rutinas guardadas en el CSV del modo actual."""
    if not os.path.exists(ruta_rutinas_csv):
        return None
    rutinas = []
    with open(ruta_rutinas_csv, mode='r', newline='') as file:
        reader = csv.reader(file)
        try:
            next(reader)
        except StopIteration:
            pass
        for row in reader:
            if row and len(row) >= 2:
                rutinas.append(row)
    return rutinas

def main():
    os.makedirs(DIRECTORIO_GUARDADO, exist_ok=True)
    os.makedirs(DIRECTORIO_RUTINAS, exist_ok=True)

    print(f"Conectando a la mano en {IP_MANO} ...")
    mano = InspireHand(ip=IP_MANO)
    posicion_memoria = None
    modo_actual_idx = 0
    # Mantiene en software la última velocidad configurada para el registro 1522
    velocidades_cfg = MAX_VELOCIDAD.copy()
    
    try:
        print("Reestableciendo a postura inicial ABIERTO...")
        enviar_parametros(mano, ABIERTO, None, velocidades_cfg, default_fuerzas=FUERZA_SEGURA, timeout=5.0)

        while True:
            archivo_csv_actual = os.path.join(DIRECTORIO_GUARDADO, ARCHIVOS_CSV[modo_actual_idx])
            archivo_rutinas_actual = os.path.join(DIRECTORIO_RUTINAS, ARCHIVOS_RUTINAS_CSV[modo_actual_idx])
            
            limpiar_pantalla()
            print("\n" + "="*65)
            print("      INTERFAZ DE CONTROL AVANZADA - RH56DFTP-2R      ")
            print("="*65)
            print(f"1. MODO ACTUAL: [{MODOS_NOMBRES[modo_actual_idx]}] (Presione 1 para cambiar)")
            print("-" * 65)
            print("2. Modo Manual (Mantiene postura actual y mueve SOLO un dedo)")
            print("3. Modo Configuración Completa (Mover todos)")
            print("4. Reestablecer a posición inicial (Mano Abierta)")
            print("5. Guardar posición actual en Memoria (RAM)")
            print("6. Ejecutar posición desde Memoria")
            print("7. Añadir posición actual a la base de datos de gestos (.csv)")
            print("8. Cargar y ejecutar UN gesto desde la base guardada")
            print("9. Crear, ejecutar y/o exportar RUTINA de gestos secuenciales")
            print("10. Importar y ejecutar RUTINA guardada (.csv)")
            print("11. Estado actual de cada dedo")
            print("0. Salir del programa")
            print("="*65)
            print(f"CSV Gestos Activo : {ARCHIVOS_CSV[modo_actual_idx]}")
            print(f"CSV Rutinas Activo: {ARCHIVOS_RUTINAS_CSV[modo_actual_idx]}")
            print("="*65)
            
            sys.stdout.flush()
            time.sleep(0.2)
            opcion = input("Seleccione una opción: ")

            if opcion == '0':
                sys.stdout.flush()
                time.sleep(0.2)
                if input("¿Seguro que desea salir? La mano se abrirá (s/n): ").lower() == 's':
                    enviar_parametros(mano, ABIERTO, MAX_FUERZA, MAX_VELOCIDAD, timeout=5.0)
                    break

            elif opcion == '1':
                modo_actual_idx = (modo_actual_idx + 1) % 4
                print(f"Modo cambiado a: {MODOS_NOMBRES[modo_actual_idx]}")
                time.sleep(0.8)

            elif opcion == '2':
                print(f"\nMODO MANUAL INCREMENTAL - Operando en: {MODOS_NOMBRES[modo_actual_idx]}")
                angulos_act, fuerzas_act = leer_estado_completo(mano)
                
                for idx, nombre in enumerate(NOMBRES):
                    lectura_info = f"[Ang: {angulos_act[idx]:4d}]"
                    if modo_actual_idx in [1, 3]:
                        lectura_info += f" [Fuerza: {fuerzas_act[idx]:4d}]"
                    print(f"{idx}. {nombre:15} {lectura_info}")
                
                try:
                    sys.stdout.flush()
                    time.sleep(0.2)
                    i = int(input("\nSeleccione el dedo a mover (0-5) o -1 para cancelar: "))
                    if i == -1: continue
                    if not (0 <= i <= 5):
                        print("⚠ Dedo inválido.")
                        time.sleep(1.5)
                        continue
                    
                    sys.stdout.flush()
                    time.sleep(0.2)
                    n_ang = int(input(f"Ingrese el Ángulo [Actual: {angulos_act[i]}] (0-1000): "))
                    n_fuerza, n_veloc = None, None
                    
                    if modo_actual_idx in [1, 3]:
                        limite_f = 1500 if i == 4 else 1000
                        sys.stdout.flush()
                        time.sleep(0.2)
                        n_fuerza = int(input(f"Ingrese la Fuerza [Actual: {fuerzas_act[i]}] (0-{limite_f}): "))
                        
                    if modo_actual_idx in [2, 3]:
                        sys.stdout.flush()
                        time.sleep(0.2)
                        n_veloc = int(input("Ingrese la Velocidad a configurar (0-1000): "))

                    mov_angulo = angulos_act.copy()
                    mov_angulo[i] = n_ang
                    
                    mov_def_fuerza = FUERZA_SEGURA.copy() if modo_actual_idx == 0 else None

                    mov_fuerza = None
                    if modo_actual_idx in [1, 3]:
                        mov_fuerza = fuerzas_act.copy()
                        mov_fuerza[i] = n_fuerza
                    
                    mov_veloc = None
                    if modo_actual_idx in [2, 3]:
                        velocidades_cfg[i] = max(0, min(1000, n_veloc))
                        mov_veloc = velocidades_cfg.copy()

                    print("\nEjecutando movimiento...")
                    enviar_parametros(mano, mov_angulo, mov_fuerza, mov_veloc, default_fuerzas=mov_def_fuerza, timeout=5.0)
                    time.sleep(1.0)
                    
                except ValueError:
                    print("⚠ Entrada no válida.")
                    time.sleep(1.5)

            elif opcion == '3':
                print(f"\nCONFIGURACIÓN COMPLETA - Operando en: {MODOS_NOMBRES[modo_actual_idx]}")
                angulos_act, fuerzas_act = leer_estado_completo(mano)
                
                nueva_postura = recoger_matriz("Ángulo", -1, angulos_act)
                nueva_fuerza, nueva_velocidad = None, None
                
                nueva_def_fuerza = FUERZA_SEGURA.copy() if modo_actual_idx == 0 else None

                if modo_actual_idx in [1, 3]:
                    nueva_fuerza = recoger_matriz("Fuerza límite", -1, fuerzas_act)
                if modo_actual_idx in [2, 3]:
                    # Sin vector_actual porque no existe sensor de velocidad
                    nueva_velocidad = recoger_matriz("Velocidad", -1, None)
                    velocidades_cfg = nueva_velocidad.copy()
                
                print("\nEjecutando movimiento...")
                time.sleep(0.5)
                enviar_parametros(mano, nueva_postura, nueva_fuerza, nueva_velocidad, default_fuerzas=nueva_def_fuerza, timeout=5.0)
                time.sleep(1.0)

            elif opcion == '4':
                print("\nReestableciendo a posición inicial...")
                velocidades_cfg = MAX_VELOCIDAD.copy()
                if modo_actual_idx == 0:
                    enviar_parametros(mano, ABIERTO, None, MAX_VELOCIDAD, default_fuerzas=FUERZA_SEGURA, timeout=5.0)
                else:
                    enviar_parametros(mano, ABIERTO, MAX_FUERZA, MAX_VELOCIDAD, timeout=5.0)
                time.sleep(1.0)

            elif opcion == '5':
                angulos_actuales, fuerzas_actuales = leer_estado_completo(mano)
                posicion_memoria = {
                    "angulos": angulos_actuales.copy(),
                    "fuerzas": fuerzas_actuales.copy(),
                    "velocidades": velocidades_cfg.copy()
                }
                print("\nEstado completo guardado en memoria RAM.")
                sys.stdout.flush()
                time.sleep(0.2)
                input("Presione ENTER para continuar...")

            elif opcion == '6':
                if posicion_memoria is not None:
                    print("\nEjecutando estado guardado desde Memoria RAM...")
                    def_f_aplicar = FUERZA_SEGURA.copy() if modo_actual_idx == 0 else None
                    f_aplicar = None if modo_actual_idx == 0 else (posicion_memoria["fuerzas"] if modo_actual_idx in [1, 3] else MAX_FUERZA.copy())
                    v_aplicar = posicion_memoria["velocidades"] if modo_actual_idx in [2, 3] else MAX_VELOCIDAD.copy()
                    
                    enviar_parametros(mano, posicion_memoria["angulos"], f_aplicar, v_aplicar, default_fuerzas=def_f_aplicar, timeout=5.0)
                    time.sleep(1.0)
                else:
                    print("\n⚠ No hay ninguna posición guardada.")
                    time.sleep(1.5)

            elif opcion == '7':
                angulos_actuales, fuerzas_actuales = leer_estado_completo(mano)
                sys.stdout.flush()
                time.sleep(0.2)
                nombre_gesto = input(f"\nIngrese un nombre para guardar en '{ARCHIVOS_CSV[modo_actual_idx]}' (ej. Pinza_Fina): ")
                
                if not nombre_gesto.strip():
                    print("⚠ El nombre no puede estar vacío.")
                    time.sleep(1.5)
                    continue

                archivo_existe = os.path.exists(archivo_csv_actual)
                
                with open(archivo_csv_actual, mode='a', newline='') as file:
                    writer = csv.writer(file)
                    if not archivo_existe:
                        writer.writerow(obtener_cabeceras(modo_actual_idx))
                    
                    fila_exportar = [nombre_gesto] + angulos_actuales
                    if modo_actual_idx == 1:
                        fila_exportar += fuerzas_actuales
                    elif modo_actual_idx == 2:
                        fila_exportar += velocidades_cfg
                    elif modo_actual_idx == 3:
                        fila_exportar += fuerzas_actuales + velocidades_cfg
                        
                    writer.writerow(fila_exportar)
                    
                print(f" ✓ Gesto '{nombre_gesto}' añadido exitosamente.")
                time.sleep(1.5)

            elif opcion in ['8', '9']:
                gestos_guardados = cargar_gestos_csv(archivo_csv_actual)
                if gestos_guardados is None:
                    print(f"\n⚠ El archivo {ARCHIVOS_CSV[modo_actual_idx]} aún no existe. Guarde una posición primero.")
                    time.sleep(2.0)
                    continue
                            
                if not gestos_guardados:
                    print(f"\n⚠ El archivo {ARCHIVOS_CSV[modo_actual_idx]} está vacío.")
                    time.sleep(1.5)
                    continue
                    
                print(f"\n--- GESTOS DISPONIBLES EN ESTE MODO ({MODOS_NOMBRES[modo_actual_idx]}) ---")
                for i, row in enumerate(gestos_guardados):
                    print(f"{i}. {row[0]}")
                    
                if opcion == '8':
                    try:
                        sys.stdout.flush()
                        time.sleep(0.2)
                        sel = int(input("\nSeleccione el número del gesto a ejecutar (o -1 para cancelar): "))
                        if sel == -1: continue
                            
                        if 0 <= sel < len(gestos_guardados):
                            gesto_row = gestos_guardados[sel]
                            print(f"\nCargando gesto '{gesto_row[0]}'...")
                            velocidades_cfg = ejecutar_gesto_desde_fila(mano, gesto_row, modo_actual_idx)
                            time.sleep(1.0)
                        else:
                            print("⚠ Selección fuera de rango.")
                            time.sleep(1.5)
                    except ValueError:
                        print("⚠ Entrada no válida.")
                        time.sleep(1.5)
                        
                elif opcion == '9':
                    sys.stdout.flush()
                    time.sleep(0.2)
                    secuencia_str = input("\nIngrese la secuencia de números separados por coma para la rutina (ej. 0,2,1,0) o ENTER para cancelar: ")
                    if not secuencia_str.strip():
                        continue
                        
                    try:
                        indices = [int(x.strip()) for x in secuencia_str.split(',') if x.strip()]
                        indices_validos = [idx for idx in indices if 0 <= idx < len(gestos_guardados)]
                        
                        if not indices_validos:
                            print("⚠ Ningún índice ingresado es válido.")
                            time.sleep(1.5)
                            continue
                            
                        if len(indices_validos) < len(indices):
                            print("⚠ Algunos índices fuera de rango fueron descartados.")
                            
                        vista_previa = " -> ".join(gestos_guardados[i][0] for i in indices_validos)
                        print(f"\nSecuencia configurada: [{vista_previa}]")
                        
                        sys.stdout.flush()
                        time.sleep(0.2)
                        desea_exportar = input(f"¿Desea exportar/guardar esta rutina en '{ARCHIVOS_RUTINAS_CSV[modo_actual_idx]}'? (s/n): ").strip().lower()
                        if desea_exportar == 's':
                            sys.stdout.flush()
                            time.sleep(0.2)
                            nombre_rutina = input("Ingrese un nombre identificador para la rutina (ej. Saludo_Completo): ").strip()
                            if not nombre_rutina:
                                nombre_rutina = f"Rutina_{int(time.time())}"
                            exportar_rutina_csv(archivo_rutinas_actual, nombre_rutina, indices_validos, gestos_guardados)
                            
                        sys.stdout.flush()
                        time.sleep(0.2)
                        desea_ejecutar = input("¿Desea ejecutar esta rutina ahora? (s/n): ").strip().lower()
                        if desea_ejecutar == 's':
                            velocidades_cfg = ejecutar_secuencia_gestos(mano, indices_validos, gestos_guardados, modo_actual_idx)
                        
                        sys.stdout.flush()
                        time.sleep(0.2)
                        input("Presione ENTER para continuar...")
                        
                    except ValueError:
                        print("⚠ Formato no válido. Asegúrese de ingresar solo números separados por coma.")
                        time.sleep(2.0)

            elif opcion == '10':
                gestos_guardados = cargar_gestos_csv(archivo_csv_actual)
                if not gestos_guardados:
                    print(f"\n⚠ No hay gestos disponibles en '{ARCHIVOS_CSV[modo_actual_idx]}' para reconstruir rutinas.")
                    time.sleep(2.0)
                    continue

                rutinas_guardadas = cargar_rutinas_csv(archivo_rutinas_actual)
                if rutinas_guardadas is None:
                    print(f"\n⚠ El archivo de rutinas '{ARCHIVOS_RUTINAS_CSV[modo_actual_idx]}' aún no existe. Exporte una rutina primero en la Opción 9.")
                    time.sleep(2.0)
                    continue

                if not rutinas_guardadas:
                    print(f"\n⚠ El archivo '{ARCHIVOS_RUTINAS_CSV[modo_actual_idx]}' está vacío.")
                    time.sleep(1.5)
                    continue

                print(f"\n--- RUTINAS DISPONIBLES EN ESTE MODO ({MODOS_NOMBRES[modo_actual_idx]}) ---")
                for i, row in enumerate(rutinas_guardadas):
                    nombre_rut = row[0]
                    indices_rut = row[1]
                    nombres_rut = row[2] if len(row) > 2 else indices_rut
                    print(f"{i}. {nombre_rut:20} | Pasos: {nombres_rut} (Índices: {indices_rut})")

                try:
                    sys.stdout.flush()
                    time.sleep(0.2)
                    sel_rut = int(input("\nSeleccione el número de la rutina a importar y ejecutar (o -1 para cancelar): "))
                    if sel_rut == -1:
                        continue

                    if 0 <= sel_rut < len(rutinas_guardadas):
                        rutina_elegida = rutinas_guardadas[sel_rut]
                        nombre_rut = rutina_elegida[0]
                        indices_rut = [int(x.strip()) for x in rutina_elegida[1].split(',') if x.strip()]
                        print(f"\nImportando rutina '{nombre_rut}' desde '{ARCHIVOS_RUTINAS_CSV[modo_actual_idx]}'...")
                        velocidades_cfg = ejecutar_secuencia_gestos(mano, indices_rut, gestos_guardados, modo_actual_idx)

                        sys.stdout.flush()
                        time.sleep(0.2)
                        input("Presione ENTER para continuar...")
                    else:
                        print("⚠ Selección fuera de rango.")
                        time.sleep(1.5)
                except ValueError:
                    print("⚠ Entrada no válida.")
                    time.sleep(1.5)
                    
            elif opcion == '11':
                sys.stdout.flush()
                time.sleep(0.2)
                angulos_act, fuerzas_act = leer_estado_completo(mano)
                for idx, nombre in enumerate(NOMBRES):
                    lectura_info = f"[Ang: {angulos_act[idx]:4d}]"
                    lectura_info += f" [Fuerza: {fuerzas_act[idx]:4d}]"
                    print(f"{idx}. {nombre:15} {lectura_info}")
                input("Presiona ENTER para continuar...")

    except KeyboardInterrupt:
        print("\nKernel Interrupt. Abriendo la mano por seguridad...")
        enviar_parametros(mano, ABIERTO, MAX_FUERZA, MAX_VELOCIDAD, timeout=5.0)
    finally:
        mano.close()
        print("Conexión TCP cerrada.")

if __name__ == "__main__":
    main()


      INTERFAZ DE CONTROL AVANZADA - RH56DFTP-2R      
1. MODO ACTUAL: [Solo Ángulos (Fuerza Segura)] (Presione 1 para cambiar)
-----------------------------------------------------------------
2. Modo Manual (Mantiene postura actual y mueve SOLO un dedo)
3. Modo Configuración Completa (Mover todos)
4. Reestablecer a posición inicial (Mano Abierta)
5. Guardar posición actual en Memoria (RAM)
6. Ejecutar posición desde Memoria
7. Añadir posición actual a la base de datos de gestos (.csv)
8. Cargar y ejecutar UN gesto desde la base guardada
9. Crear, ejecutar y/o exportar RUTINA de gestos secuenciales
10. Importar y ejecutar RUTINA guardada (.csv)
11. Estado actual de cada dedo
0. Salir del programa
CSV Gestos Activo : gestos_0_solo_angulos.csv
CSV Rutinas Activo: rutinas_0_solo_angulos.csv


Seleccione una opción:  0
¿Seguro que desea salir? La mano se abrirá (s/n):  s


 ✓ Gesto completado en 0.83s.
Conexión TCP cerrada.
